[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_07/1_algorithms.ipynb)

# Day 7: Algorithms (mock exam)

**Mock exam rules.**
- Start one 60 minute timer for the whole notebook and do not stop it between questions.
- Do not open any hint or solution until the timer ends. Then check, and read the solutions.
- For each problem, first say out loud: the pattern, the approach, and the time and space complexity. Then code.
- Run your own extra test (an edge case you chose) before you move on, like you would in a real exam.
- After the timer: score yourself (solved, solved after the timer, not solved) and log every miss.

**How to work:** Name the pattern before you code. Stuck 15 min: open Hint 1. Stuck 25 min: Hint 2, then the solution. Say the time and space complexity out loud. Re-type the solution from memory if you needed it.

**Today (60 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Smash the two heaviest | easy | 20 |
| Q2 | The value that wins the vote | easy | 20 |
| Q3 | Merge into the bigger array | easy | 20 |

**Handbook:** [Algorithms](https://mahsa-agz.github.io/ds-handbook/#algo), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import copy

def check(fn, cases, key=None):
    """Runs fn on each (args, expected) case and prints PASS or FAIL. key: normalise outputs (e.g. sorted)."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            got = "error: " + repr(e)
        good = (key(got) == key(want)) if (key and not isinstance(got, str)) else got == want
        ok += good
        shown = ", ".join(repr(a) for a in args)
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}({shown}) -> {got!r}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

---
### Q1. Smash the two heaviest

*easy, about 20 min*

You have stones with positive integer weights. Each turn, take the two heaviest stones `x <= y` and smash
them: if `x == y` both are destroyed, otherwise only a stone of weight `y - x` remains. When at most one stone is
left, return its weight (or `0` if none is left).

Examples:
- `[2, 7, 4, 1, 8, 1]` -> `1` (8 and 7 leave 1, then 4 and 2 leave 2, then 2 and 1 leave 1, then 1 and 1 vanish,
  one stone of weight 1 remains)
- `[10, 4, 2, 10]` -> `2`
- `[2, 2]` -> `0`

Follow-up to answer out loud: what is the cost if you re-sort the list every turn instead?

*Source: [LeetCode 1046](https://leetcode.com/problems/last-stone-weight/)*

In [ ]:
def last_stone_weight(stones):
    pass

# tests
check(last_stone_weight, [
    ([2, 7, 4, 1, 8, 1], 1),
    ([1], 1),
    ([2, 2], 0),
    ([10, 4, 2, 10], 2),
    ([3, 7, 2], 2),
    ([9, 3, 2, 10], 0),
    ([1, 3], 2),
])

<details><summary><b>Hint 1</b></summary>

Signal: you repeatedly need the largest items of a collection that keeps changing. Pattern: **heap (priority queue)**; Python's `heapq` is a min-heap, so store negative weights.

</details>

<details><summary><b>Hint 2</b></summary>

1. `heap = [-s for s in stones]`, then `heapq.heapify(heap)`.
2. While more than one stone: pop `y` (largest) and `x` (second largest), as positive numbers.
3. If `y != x`, push `-(y - x)` back.
4. Return `-heap[0]` if the heap is not empty, else 0.

</details>

<details><summary><b>Solution</b></summary>

```python
import heapq

def last_stone_weight(stones):
    heap = [-s for s in stones]          # negate: heapq is a min-heap
    heapq.heapify(heap)
    while len(heap) > 1:
        y = -heapq.heappop(heap)         # heaviest
        x = -heapq.heappop(heap)         # second heaviest
        if y != x:
            heapq.heappush(heap, -(y - x))
    return -heap[0] if heap else 0
```

**Why:** Each turn needs the two largest values of a set that changes. A heap gives the maximum in O(log n) per pop or push, so n turns cost O(n log n). Re-sorting every turn (the follow-up) costs O(n log n) per turn, so O(n^2 log n) in total.

**Complexity:** O(n log n) time, O(n) space (O(1) extra if you heapify a negated copy in place).

**Common mistakes:** Forgetting to negate (pops the lightest stones). Pushing a 0 stone back when the two weights are equal. Returning `heap[0]` without flipping the sign.

**Learn more:** [algo-heap](https://mahsa-agz.github.io/ds-handbook/#algo-heap)

</details>

---
### Q2. The value that wins the vote

*easy, about 20 min*

`nums` has length `n` and one value appears more than `n / 2` times (it is guaranteed to exist). Return
that value. First give an O(n) time solution, then the follow-up: can you do it with O(1) extra space?

Examples:
- `[3, 2, 3]` -> `3`
- `[2, 2, 1, 1, 1, 2, 2]` -> `2`
- `[-1, -1, -1, 4]` -> `-1`

*Source: [LeetCode 169](https://leetcode.com/problems/majority-element/)*

In [ ]:
def majority_element(nums):
    pass

# tests
check(majority_element, [
    ([3, 2, 3], 3),
    ([2, 2, 1, 1, 1, 2, 2], 2),
    ([1], 1),
    ([6, 5, 5], 5),
    ([-1, -1, -1, 4], -1),
    ([4, 4, 7, 7, 7, 4, 4], 4),
])

<details><summary><b>Hint 1</b></summary>

Signal: count how often each value appears. Pattern: **hashing** (a count dict). Follow-up: Boyer-Moore voting, where different values cancel each other out.

</details>

<details><summary><b>Hint 2</b></summary>

Count dict: count every value, return the one with count > n // 2.
Voting (O(1) space):
1. `candidate = None`, `votes = 0`.
2. For each `x`: if `votes == 0`, set `candidate = x`. Then `votes += 1` if `x == candidate` else `votes -= 1`.
3. Return `candidate` (valid because a majority is guaranteed).

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import Counter

def majority_element(nums):              # O(n) time, O(n) space
    count = Counter(nums)
    return max(count, key=count.get)

def majority_element_vote(nums):         # follow-up: O(n) time, O(1) space
    candidate, votes = None, 0
    for x in nums:
        if votes == 0:
            candidate = x
        votes += 1 if x == candidate else -1
    return candidate

print(majority_element_vote([2, 2, 1, 1, 1, 2, 2]))   # 2
```

**Why:** The dict answer is the direct one. For the vote: pair each majority copy with a different value and cancel them; because the majority has more than half of the items, some copies always survive, so the final candidate is the majority. If the majority were not guaranteed, you would need a second pass to count the candidate.

**Complexity:** Count dict: O(n) time, O(n) space. Voting: O(n) time, O(1) space. (Sorting and taking `nums[n // 2]` is O(n log n).)

**Common mistakes:** Returning the candidate without a verification pass when the majority is not guaranteed. Using `nums.count(x)` for every x (O(n^2)).

**Learn more:** [algo-hashing](https://mahsa-agz.github.io/ds-handbook/#algo-hashing)

</details>

---
### Q3. Merge into the bigger array

*easy, about 20 min*

`nums1` has length `m + n`: its first `m` values are sorted, and the last `n` slots are zeros (free
space). `nums2` has `n` sorted values. Merge `nums2` into `nums1` **in place** so that `nums1` is fully sorted.
Return nothing. Target: O(m + n) time and O(1) extra space.

Examples:
- `nums1 = [1, 2, 3, 0, 0, 0], m = 3, nums2 = [2, 5, 6], n = 3` -> `nums1 = [1, 2, 2, 3, 5, 6]`
- `nums1 = [0], m = 0, nums2 = [1], n = 1` -> `nums1 = [1]`
- `nums1 = [4, 5, 0, 0], m = 2, nums2 = [1, 2], n = 2` -> `nums1 = [1, 2, 4, 5]`

*Source: [LeetCode 88](https://leetcode.com/problems/merge-sorted-array/)*

In [ ]:
def merge(nums1, m, nums2, n):
    pass

# tests
def merged(nums1, m, nums2, n):
    merge(nums1, m, nums2, n)
    return nums1

check(merged, [
    (([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3), [1, 2, 2, 3, 5, 6]),
    (([1], 1, [], 0), [1]),
    (([0], 0, [1], 1), [1]),
    (([4, 5, 0, 0], 2, [1, 2], 2), [1, 2, 4, 5]),
    (([2, 0], 1, [1], 1), [1, 2]),
    (([1, 3, 5, 0, 0], 3, [3, 4], 2), [1, 3, 3, 4, 5]),
])

<details><summary><b>Hint 1</b></summary>

Signal: two sorted arrays and free space only at the end of one of them. Pattern: **two pointers, filling from the back**.

</details>

<details><summary><b>Hint 2</b></summary>

1. `i = m - 1` (last real value in nums1), `j = n - 1` (last in nums2), `w = m + n - 1` (write slot).
2. While `j >= 0`: if `i >= 0` and `nums1[i] > nums2[j]`, write `nums1[i]` and move `i`; else write `nums2[j]`
   and move `j`. Move `w` each time.
3. When `nums2` is used up, the rest of `nums1` is already in place.

</details>

<details><summary><b>Solution</b></summary>

```python
def merge(nums1, m, nums2, n):
    i, j, w = m - 1, n - 1, m + n - 1
    while j >= 0:
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[w] = nums1[i]
            i -= 1
        else:
            nums1[w] = nums2[j]
            j -= 1
        w -= 1
```

**Why:** Filling from the front would overwrite values of `nums1` that are not merged yet. The back of `nums1` is free, and the largest remaining value always goes there, so writing from the back never destroys unread data. The loop can stop when `nums2` is empty because the left part of `nums1` is already sorted in place.

**Complexity:** O(m + n) time, O(1) space.

**Common mistakes:** Merging from the front (overwrites data). Looping `while i >= 0` instead of `while j >= 0` (leftover `nums2` values never get copied). `nums1 = sorted(...)` rebinds the name and does not change the caller's list (and is O((m + n) log(m + n))).

**Learn more:** [algo-two-pointers](https://mahsa-agz.github.io/ds-handbook/#algo-two-pointers)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [SQL](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_07/2_sql.ipynb)